In [ ]:
import numpy as np
import pandas as pd
from numba import njit
from scipy.stats import norm
from scipy.interpolate import RBFInterpolator

In [ ]:
btc_spot = pd.read_parquet("btc_spot_historical_price.parquet")
btc = btc_spot.copy()

btc["timestamp"] = pd.to_datetime(btc["timestamp"], utc=True)

btc = btc.rename(columns={
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
})

btc = btc[
    [
        "timestamp",
        "open",
        "high",
        "low",
        "close",
    ]
].sort_values("timestamp")

btc = btc.drop_duplicates("timestamp")
btc = btc.reset_index(drop=True)
btc

,timestamp,open,high,low,close
0,2025-12-04 08:00:00+00:00,90990.23,99000.00,90990.23,93306.00
1,2025-12-04 08:15:00+00:00,93227.98,93241.93,93174.88,93175.05
2,2025-12-04 08:30:00+00:00,93125.13,93182.61,93077.14,93182.61
3,2025-12-04 08:45:00+00:00,93182.61,93182.61,93182.61,93182.61
4,2025-12-04 09:00:00+00:00,93428.30,93428.30,93428.06,93428.06
...,...,...,...,...,...
25787,2026-08-29 22:45:00+00:00,78134.23,78134.23,78134.23,78134.23
25788,2026-08-29 23:00:00+00:00,78249.03,78288.00,78249.03,78263.36
25789,2026-08-29 23:15:00+00:00,78305.94,78305.94,78305.94,78305.94
25790,2026-08-29 23:30:00+00:00,78265.53,78267.66,78265.53,78267.66


In [ ]:
eth_spot = pd.read_parquet("eth_spot_historical_price.parquet")
eth = eth_spot.copy()

eth["timestamp"] = pd.to_datetime(eth["timestamp"], utc=True)

eth = eth.rename(columns={
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
})

eth = eth[
    [
        "timestamp",
        "open",
        "high",
        "low",
        "close",
    ]
].sort_values("timestamp")

eth = eth.drop_duplicates("timestamp")
eth = eth.reset_index(drop=True)
eth

,timestamp,open,high,low,close
0,2025-10-01 00:00:00+00:00,4145.15,4151.41,4144.65,4151.41
1,2025-10-01 00:05:00+00:00,4151.41,4152.37,4148.33,4149.14
2,2025-10-01 00:10:00+00:00,4149.14,4149.25,4144.35,4145.84
3,2025-10-01 00:15:00+00:00,4145.85,4147.04,4140.17,4144.72
4,2025-10-01 00:20:00+00:00,4144.72,4145.35,4136.04,4137.47
...,...,...,...,...,...
87547,2026-07-31 23:35:00+00:00,1864.16,1864.18,1863.34,1863.92
87548,2026-07-31 23:40:00+00:00,1863.91,1864.12,1863.27,1863.61
87549,2026-07-31 23:45:00+00:00,1863.61,1863.74,1862.19,1862.19
87550,2026-07-31 23:50:00+00:00,1862.20,1863.28,1860.95,1861.76


In [ ]:
btc_options = pd.read_parquet("btc_options_mark_iv.parquet")
btc_options

,instId,side,sz,px,source,tradeId,ts,uly,settleCcy,stk,...,expTime,ctVal,timestamp,expiry_dt,strike,price,size,T,spot,iv
0,BTC-USD-261225-120000-C,sell,257,0.0935,0,15,1768269960036,BTC-USD,BTC,120000,...,1798185600000,1,2026-01-13 02:06:00.036000+00:00,2026-12-25 08:00:00+00:00,120000,0.0935,257,0.947969,91290.13,0.486159
1,BTC-USD-261225-120000-C,sell,109,0.0965,0,16,1768299214032,BTC-USD,BTC,120000,...,1798185600000,1,2026-01-13 10:13:34.032000+00:00,2026-12-25 08:00:00+00:00,120000,0.0965,109,0.947042,92238.62,0.486803
2,BTC-USD-261225-160000-C,buy,238,0.0365,0,13,1768299348246,BTC-USD,BTC,160000,...,1798185600000,1,2026-01-13 10:15:48.246000+00:00,2026-12-25 08:00:00+00:00,160000,0.0365,238,0.947038,92217.77,0.484661
3,BTC-USD-261225-240000-C,sell,30,0.008,0,2,1768303125996,BTC-USD,BTC,240000,...,1798185600000,1,2026-01-13 11:18:45.996000+00:00,2026-12-25 08:00:00+00:00,240000,0.0080,30,0.946918,92149.42,0.509809
4,BTC-USD-261225-160000-C,sell,10,0.037,0,14,1768308858706,BTC-USD,BTC,160000,...,1798185600000,1,2026-01-13 12:54:18.706000+00:00,2026-12-25 08:00:00+00:00,160000,0.0370,10,0.946737,92122.57,0.487295
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100906,BTC-USD-260823-72750-P,sell,150,0.009,0,1,1787270527848,BTC-USD,BTC,72750,...,1787472000000,1,2026-08-21 00:02:07.848000+00:00,2026-08-23 08:00:00+00:00,72750,0.0090,150,0.006384,73107.36,0.354619
100907,BTC-USD-260822-76000-C,sell,100,0.0013,0,49,1787270604277,BTC-USD,BTC,76000,...,1787385600000,1,2026-08-21 00:03:24.277000+00:00,2026-08-22 08:00:00+00:00,76000,0.0013,100,0.003644,73107.36,0.486255
100908,BTC-USD-260925-95000-C,sell,1,0.0023,0,1092,1787270624154,BTC-USD,BTC,95000,...,1790323200000,1,2026-08-21 00:03:44.154000+00:00,2026-09-25 08:00:00+00:00,95000,0.0023,1,0.096730,73107.36,0.465619
100909,BTC-USD-260822-75000-C,sell,72,0.0026,0,39,1787270682538,BTC-USD,BTC,75000,...,1787385600000,1,2026-08-21 00:04:42.538000+00:00,2026-08-22 08:00:00+00:00,75000,0.0026,72,0.003642,73107.36,0.452573


In [51]:
eth_options = pd.read_parquet("eth_options_mark_iv.parquet")
eth_options

,instId,tradeId,px,sz,side,ts,source,uly,settleCcy,stk,...,strike,price,size,T,Open,High,Low,Close,Volume,iv
0,ETH-USD-260925-3400-C,31,0.1895,13,buy,1768134034495,0,ETH-USD,ETH,3400,...,3400,0.1895,13,0.703132,3110.13,3110.13,3108.04,3109.12,228.5105,0.675609
1,ETH-USD-260925-3400-C,32,0.1905,10,sell,1768187045028,0,ETH-USD,ETH,3400,...,3400,0.1905,10,0.701452,3157.65,3165.00,3153.59,3156.56,2268.3309,0.662814
2,ETH-USD-260925-4000-C,33,0.134,10,sell,1768188620057,0,ETH-USD,ETH,4000,...,4000,0.1340,10,0.701403,3164.00,3165.63,3157.59,3160.46,1194.9698,0.656279
3,ETH-USD-260925-4000-C,34,0.134,17,sell,1768203449572,0,ETH-USD,ETH,4000,...,4000,0.1340,17,0.700933,3150.25,3150.47,3142.88,3146.85,1717.2628,0.660477
4,ETH-USD-260925-2200-P,21,0.0715,100,buy,1768228097168,0,ETH-USD,ETH,2200,...,2200,0.0715,100,0.700152,3076.07,3079.67,3068.43,3077.99,2730.0183,0.639355
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78573,ETH-USD-260831-2800-C,437,0.0002,3,buy,1788077146275,0,ETH-USD,ETH,2800,...,2800,0.0002,3,0.002727,1861.76,1862.89,1861.75,1862.60,94.6250,2.897568
78574,ETH-USD-260831-2800-C,438,0.0002,3,buy,1788077146338,0,ETH-USD,ETH,2800,...,2800,0.0002,3,0.002727,1861.76,1862.89,1861.75,1862.60,94.6250,2.897569
78575,ETH-USD-260831-2480-C,33,0.0039,27,buy,1788077153287,0,ETH-USD,ETH,2480,...,2480,0.0039,27,0.002727,1861.76,1862.89,1861.75,1862.60,94.6250,3.286364
78576,ETH-USD-260831-2480-C,34,0.0039,13,buy,1788077153287,0,ETH-USD,ETH,2480,...,2480,0.0039,13,0.002727,1861.76,1862.89,1861.75,1862.60,94.6250,3.286364


In [91]:
btc_df = pd.merge_asof(
    btc_options.sort_values("timestamp"),
    btc[["timestamp", "high"]].sort_values("timestamp"),
    on="timestamp",
    direction="backward",
    tolerance=pd.Timedelta(minutes=15),
)

btc_df

,instId,side,sz,px,source,tradeId,ts,uly,settleCcy,stk,...,ctVal,timestamp,expiry_dt,strike,price,size,T,spot,iv,high
0,BTC-USD-261225-120000-C,sell,257,0.0935,0,15,1768269960036,BTC-USD,BTC,120000,...,1,2026-01-13 02:06:00.036000+00:00,2026-12-25 08:00:00+00:00,120000,0.0935,257,0.947969,91290.13,0.486159,91207.89
1,BTC-USD-261225-120000-C,sell,109,0.0965,0,16,1768299214032,BTC-USD,BTC,120000,...,1,2026-01-13 10:13:34.032000+00:00,2026-12-25 08:00:00+00:00,120000,0.0965,109,0.947042,92238.62,0.486803,92260.60
2,BTC-USD-261225-160000-C,buy,238,0.0365,0,13,1768299348246,BTC-USD,BTC,160000,...,1,2026-01-13 10:15:48.246000+00:00,2026-12-25 08:00:00+00:00,160000,0.0365,238,0.947038,92217.77,0.484661,92170.71
3,BTC-USD-261225-240000-C,sell,30,0.008,0,2,1768303125996,BTC-USD,BTC,240000,...,1,2026-01-13 11:18:45.996000+00:00,2026-12-25 08:00:00+00:00,240000,0.0080,30,0.946918,92149.42,0.509809,92163.02
4,BTC-USD-261225-160000-C,sell,10,0.037,0,14,1768308858706,BTC-USD,BTC,160000,...,1,2026-01-13 12:54:18.706000+00:00,2026-12-25 08:00:00+00:00,160000,0.0370,10,0.946737,92122.57,0.487295,92084.15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100906,BTC-USD-260823-72750-P,sell,150,0.009,0,1,1787270527848,BTC-USD,BTC,72750,...,1,2026-08-21 00:02:07.848000+00:00,2026-08-23 08:00:00+00:00,72750,0.0090,150,0.006384,73107.36,0.354619,73423.20
100907,BTC-USD-260822-76000-C,sell,100,0.0013,0,49,1787270604277,BTC-USD,BTC,76000,...,1,2026-08-21 00:03:24.277000+00:00,2026-08-22 08:00:00+00:00,76000,0.0013,100,0.003644,73107.36,0.486255,73423.20
100908,BTC-USD-260925-95000-C,sell,1,0.0023,0,1092,1787270624154,BTC-USD,BTC,95000,...,1,2026-08-21 00:03:44.154000+00:00,2026-09-25 08:00:00+00:00,95000,0.0023,1,0.096730,73107.36,0.465619,73423.20
100909,BTC-USD-260822-75000-C,sell,72,0.0026,0,39,1787270682538,BTC-USD,BTC,75000,...,1,2026-08-21 00:04:42.538000+00:00,2026-08-22 08:00:00+00:00,75000,0.0026,72,0.003642,73107.36,0.452573,73423.20


In [52]:
eth_df = pd.merge_asof(
    eth_options.sort_values("timestamp"),
    eth[["timestamp", "high"]].sort_values("timestamp"),
    on="timestamp",
    direction="backward",
    tolerance=pd.Timedelta(minutes=15),
)

eth_df

,instId,tradeId,px,sz,side,ts,source,uly,settleCcy,stk,...,price,size,T,Open,High,Low,Close,Volume,iv,high
0,ETH-USD-260925-3400-C,31,0.1895,13,buy,1768134034495,0,ETH-USD,ETH,3400,...,0.1895,13,0.703132,3110.13,3110.13,3108.04,3109.12,228.5105,0.675609,3110.13
1,ETH-USD-260925-3400-C,32,0.1905,10,sell,1768187045028,0,ETH-USD,ETH,3400,...,0.1905,10,0.701452,3157.65,3165.00,3153.59,3156.56,2268.3309,0.662814,3165.00
2,ETH-USD-260925-4000-C,33,0.134,10,sell,1768188620057,0,ETH-USD,ETH,4000,...,0.1340,10,0.701403,3164.00,3165.63,3157.59,3160.46,1194.9698,0.656279,3165.63
3,ETH-USD-260925-4000-C,34,0.134,17,sell,1768203449572,0,ETH-USD,ETH,4000,...,0.1340,17,0.700933,3150.25,3150.47,3142.88,3146.85,1717.2628,0.660477,3150.47
4,ETH-USD-260925-2200-P,21,0.0715,100,buy,1768228097168,0,ETH-USD,ETH,2200,...,0.0715,100,0.700152,3076.07,3079.67,3068.43,3077.99,2730.0183,0.639355,3079.67
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78573,ETH-USD-260831-2800-C,437,0.0002,3,buy,1788077146275,0,ETH-USD,ETH,2800,...,0.0002,3,0.002727,1861.76,1862.89,1861.75,1862.60,94.6250,2.897568,NaN
78574,ETH-USD-260831-2800-C,438,0.0002,3,buy,1788077146338,0,ETH-USD,ETH,2800,...,0.0002,3,0.002727,1861.76,1862.89,1861.75,1862.60,94.6250,2.897569,NaN
78575,ETH-USD-260831-2480-C,34,0.0039,13,buy,1788077153287,0,ETH-USD,ETH,2480,...,0.0039,13,0.002727,1861.76,1862.89,1861.75,1862.60,94.6250,3.286364,NaN
78576,ETH-USD-260831-2480-C,33,0.0039,27,buy,1788077153287,0,ETH-USD,ETH,2480,...,0.0039,27,0.002727,1861.76,1862.89,1861.75,1862.60,94.6250,3.286364,NaN


In [92]:
def generate_synthetic_contracts(
    timestamps,
    btc,
    barriers,
    horizons,
    sample_frequency="1h",
):
    """
    Generate synthetic BTC barrier-touch contracts.

    Each synthetic contract asks:

        UP:
            Will BTC touch barrier S before expiry?

        DOWN:
            Will BTC touch barrier S before expiry?

    Parameters
    ----------
    timestamps : pd.Series / array-like
        Timestamps at which an options surface/model probability
        is available.

    btc : pd.DataFrame
        BTC OHLC data with columns:
            timestamp
            open
            high
            low
            close

    barriers : list[float]
        Explicit BTC price barriers to test.
        Example:
            [50000, 60000, 70000, ..., 200000]

    horizons : dict
        Example:
            {
                "7d": pd.Timedelta(days=7),
                "30d": pd.Timedelta(days=30),
                "90d": pd.Timedelta(days=90),
            }

    sample_frequency : str
        Frequency at which to create synthetic contracts.
        Example:
            "1h", "4h", "1d"

    Returns
    -------
    pd.DataFrame
        One row per synthetic contract.
    """

    # --------------------------------------------------------
    # Clean BTC data
    # --------------------------------------------------------

    btc = btc.copy()

    btc["timestamp"] = pd.to_datetime(
        btc["timestamp"],
        utc=True
    )

    btc = btc.rename(columns={
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
    })

    btc = btc[
        [
            "timestamp",
            "open",
            "high",
            "low",
            "close",
        ]
    ].sort_values("timestamp")

    btc = btc.drop_duplicates(
        subset=["timestamp"]
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # Decision timestamps
    # --------------------------------------------------------

    decision_times = pd.DataFrame({
        "timestamp": pd.to_datetime(
            timestamps,
            utc=True
        )
    })

    decision_times = (
        decision_times
        .drop_duplicates()
        .sort_values("timestamp")
    )

    # Sample timestamps.
    #
    # This avoids creating a contract at every options
    # observation if you have many observations per hour.
    decision_times = (
        decision_times
        .set_index("timestamp")
        .resample(sample_frequency)
        .first()
        .dropna()
        .reset_index()
    )

    # --------------------------------------------------------
    # Get BTC spot available at decision time
    # --------------------------------------------------------

    decision_times = pd.merge_asof(
        decision_times.sort_values("timestamp"),
        btc[
            ["timestamp", "high"]
        ].sort_values("timestamp"),
        on="timestamp",
        direction="backward",
        tolerance=pd.Timedelta(minutes=30),
    )

    decision_times = decision_times.dropna(
        subset=["high"]
    )

    # --------------------------------------------------------
    # Generate contracts
    # --------------------------------------------------------

    contracts = []

    btc_last_timestamp = btc["timestamp"].max()

    for _, row in decision_times.iterrows():

        timestamp = row["timestamp"]
        spot = float(row["high"])

        for horizon_name, horizon in horizons.items():

            expiry = timestamp + horizon

            # Cannot evaluate this contract because
            # we don't have the complete future BTC path.
            if expiry > btc_last_timestamp:
                continue

            for barrier in barriers:

                barrier = float(barrier)

                # ------------------------------------------------
                # UP TOUCH
                # ------------------------------------------------
                #
                # "Will BTC touch S before expiry?"
                #
                # Only valid as an UP contract if BTC is
                # currently below S.
                #

                if spot < barrier:

                    contracts.append({
                        "timestamp": timestamp,
                        "spot": spot,
                        "direction": "up",
                        "barrier": barrier,
                        "horizon": horizon_name,
                        "expiry": expiry,
                    })

                # ------------------------------------------------
                # DOWN TOUCH
                # ------------------------------------------------
                #
                # "Will BTC touch S before expiry?"
                #
                # Only valid as a DOWN contract if BTC is
                # currently above S.
                #

                elif spot > barrier:

                    contracts.append({
                        "timestamp": timestamp,
                        "spot": spot,
                        "direction": "down",
                        "barrier": barrier,
                        "horizon": horizon_name,
                        "expiry": expiry,
                    })

                # ------------------------------------------------
                # If spot == barrier
                # ------------------------------------------------
                #
                # The barrier has already been touched, so
                # don't create a new contract.
                #

    return pd.DataFrame(contracts)

btc_barriers = [
    50000,
    60000,
    70000,
    80000,
    90000,
    100000,
    110000,
    120000,
    130000,
    140000,
    150000,
    160000,
    170000,
    180000,
    190000,
    200000,
]

btc_horizons = {
    "7d": pd.Timedelta(days=7),
    "30d": pd.Timedelta(days=30),
    "90d": pd.Timedelta(days=90),
    "180d": pd.Timedelta(days=180),
}

btc_contracts = generate_synthetic_contracts(
    timestamps = btc_options["timestamp"],
    btc = btc_spot,
    barriers=btc_barriers,
    horizons=btc_horizons,
    sample_frequency="1h"
)

btc_contracts

,timestamp,spot,direction,barrier,horizon,expiry
0,2026-01-13 02:00:00+00:00,91207.89,down,50000.0,7d,2026-01-20 02:00:00+00:00
1,2026-01-13 02:00:00+00:00,91207.89,down,60000.0,7d,2026-01-20 02:00:00+00:00
2,2026-01-13 02:00:00+00:00,91207.89,down,70000.0,7d,2026-01-20 02:00:00+00:00
3,2026-01-13 02:00:00+00:00,91207.89,down,80000.0,7d,2026-01-20 02:00:00+00:00
4,2026-01-13 02:00:00+00:00,91207.89,down,90000.0,7d,2026-01-20 02:00:00+00:00
...,...,...,...,...,...,...
232968,2026-08-21 00:00:00+00:00,73423.20,up,160000.0,7d,2026-08-28 00:00:00+00:00
232969,2026-08-21 00:00:00+00:00,73423.20,up,170000.0,7d,2026-08-28 00:00:00+00:00
232970,2026-08-21 00:00:00+00:00,73423.20,up,180000.0,7d,2026-08-28 00:00:00+00:00
232971,2026-08-21 00:00:00+00:00,73423.20,up,190000.0,7d,2026-08-28 00:00:00+00:00


In [53]:
eth_barriers = [
    1000,
    1200,
    1400,
    1600,
    1800,
    2000,
    2200,
    2400,
    2600,
    2800,
    3000,
    3200,
    3400,
    3600,
    3800,
    4000,
]

eth_horizons = {
    "7d": pd.Timedelta(days=7),
    "30d": pd.Timedelta(days=30),
    "90d": pd.Timedelta(days=90),
    "180d": pd.Timedelta(days=180),
}

eth_contracts = generate_synthetic_contracts(
    timestamps = eth_options["timestamp"],
    btc = eth_spot,
    barriers=eth_barriers,
    horizons=eth_horizons,
    sample_frequency="1h"
)

eth_contracts

,timestamp,spot,direction,barrier,horizon,expiry
0,2026-01-11 12:00:00+00:00,3107.69,down,1000.0,7d,2026-01-18 12:00:00+00:00
1,2026-01-11 12:00:00+00:00,3107.69,down,1200.0,7d,2026-01-18 12:00:00+00:00
2,2026-01-11 12:00:00+00:00,3107.69,down,1400.0,7d,2026-01-18 12:00:00+00:00
3,2026-01-11 12:00:00+00:00,3107.69,down,1600.0,7d,2026-01-18 12:00:00+00:00
4,2026-01-11 12:00:00+00:00,3107.69,down,1800.0,7d,2026-01-18 12:00:00+00:00
...,...,...,...,...,...,...
191609,2026-07-24 23:00:00+00:00,1860.82,up,3200.0,7d,2026-07-31 23:00:00+00:00
191610,2026-07-24 23:00:00+00:00,1860.82,up,3400.0,7d,2026-07-31 23:00:00+00:00
191611,2026-07-24 23:00:00+00:00,1860.82,up,3600.0,7d,2026-07-31 23:00:00+00:00
191612,2026-07-24 23:00:00+00:00,1860.82,up,3800.0,7d,2026-07-31 23:00:00+00:00


In [ ]:
@njit
def calculate_touch_numba(
    contract_ts,
    expiry,
    barriers,
    directions,
    spot_ts,
    highs,
    lows
):
    n = len(contract_ts)
    result = np.empty(n, dtype=np.int8)

    for i in range(n):

        # First BTC candle strictly AFTER contract timestamp
        start = np.searchsorted(
            spot_ts,
            contract_ts[i],
            side="right"
        )

        # Last BTC candle <= expiry
        end = np.searchsorted(
            spot_ts,
            expiry[i],
            side="right"
        )

        if start >= end:
            result[i] = -1
            continue

        barrier = barriers[i]

        if directions[i] == 1:  # UP

            max_high = np.max(highs[start:end])
            result[i] = 1 if max_high >= barrier else 0

        else:  # DOWN

            min_low = np.min(lows[start:end])
            result[i] = 1 if min_low <= barrier else 0

    return result


def calculate_realized_touch(contracts, spot):

    contracts = contracts.copy()
    spot = spot.copy()

    spot = spot.sort_values("timestamp").reset_index(drop=True)

    # Normalize timestamps to UTC
    contracts["timestamp"] = pd.to_datetime(
        contracts["timestamp"], utc=True
    )

    contracts["expiry"] = pd.to_datetime(
        contracts["expiry"], utc=True
    )

    spot["timestamp"] = pd.to_datetime(
        spot["timestamp"], utc=True
    )

    # Explicitly normalize all timestamps to milliseconds
    contract_ts = (
        contracts["timestamp"]
        .dt.as_unit("ms")
        .astype("int64")
        .to_numpy()
    )

    expiry = (
        contracts["expiry"]
        .dt.as_unit("ms")
        .astype("int64")
        .to_numpy()
    )

    spot_ts = (
        spot["timestamp"]
        .dt.as_unit("ms")
        .astype("int64")
        .to_numpy()
    )

    barriers = contracts["barrier"].to_numpy(
        dtype=np.float64
    )

    directions = (
        contracts["direction"]
        .eq("up")
        .to_numpy(dtype=np.int8)
    )

    highs = spot["High"].to_numpy(dtype=np.float64)
    lows = spot["Low"].to_numpy(dtype=np.float64)

    result = calculate_touch_numba(
        contract_ts,
        expiry,
        barriers,
        directions,
        spot_ts,
        highs,
        lows
    )

    contracts["realized_touch"] = result

    return contracts

In [94]:
btc_realized_df = calculate_realized_touch(btc_contracts, btc_spot)
btc_realized_df

,timestamp,spot,direction,barrier,horizon,expiry,realized_touch
0,2026-01-13 02:00:00+00:00,91207.89,down,50000.0,7d,2026-01-20 02:00:00+00:00,0
1,2026-01-13 02:00:00+00:00,91207.89,down,60000.0,7d,2026-01-20 02:00:00+00:00,0
2,2026-01-13 02:00:00+00:00,91207.89,down,70000.0,7d,2026-01-20 02:00:00+00:00,0
3,2026-01-13 02:00:00+00:00,91207.89,down,80000.0,7d,2026-01-20 02:00:00+00:00,0
4,2026-01-13 02:00:00+00:00,91207.89,down,90000.0,7d,2026-01-20 02:00:00+00:00,0
...,...,...,...,...,...,...,...
232968,2026-08-21 00:00:00+00:00,73423.20,up,160000.0,7d,2026-08-28 00:00:00+00:00,0
232969,2026-08-21 00:00:00+00:00,73423.20,up,170000.0,7d,2026-08-28 00:00:00+00:00,0
232970,2026-08-21 00:00:00+00:00,73423.20,up,180000.0,7d,2026-08-28 00:00:00+00:00,0
232971,2026-08-21 00:00:00+00:00,73423.20,up,190000.0,7d,2026-08-28 00:00:00+00:00,0


In [81]:
eth_realized_df = calculate_realized_touch(eth_contracts, eth_spot)
eth_realized_df

,timestamp,spot,direction,barrier,horizon,expiry,realized_touch
0,2026-01-11 12:00:00+00:00,3107.69,down,1000.0,7d,2026-01-18 12:00:00+00:00,0
1,2026-01-11 12:00:00+00:00,3107.69,down,1200.0,7d,2026-01-18 12:00:00+00:00,0
2,2026-01-11 12:00:00+00:00,3107.69,down,1400.0,7d,2026-01-18 12:00:00+00:00,0
3,2026-01-11 12:00:00+00:00,3107.69,down,1600.0,7d,2026-01-18 12:00:00+00:00,0
4,2026-01-11 12:00:00+00:00,3107.69,down,1800.0,7d,2026-01-18 12:00:00+00:00,0
...,...,...,...,...,...,...,...
191609,2026-07-24 23:00:00+00:00,1860.82,up,3200.0,7d,2026-07-31 23:00:00+00:00,0
191610,2026-07-24 23:00:00+00:00,1860.82,up,3400.0,7d,2026-07-31 23:00:00+00:00,0
191611,2026-07-24 23:00:00+00:00,1860.82,up,3600.0,7d,2026-07-31 23:00:00+00:00,0
191612,2026-07-24 23:00:00+00:00,1860.82,up,3800.0,7d,2026-07-31 23:00:00+00:00,0


In [ ]:
# ============================================================
# MODEL
# ============================================================
class OptionSurface:
    # --------------------------------------------------------
    # Build variance surface
    # --------------------------------------------------------

    def build_variance_surface(self, weighted_spot, surface_df):

        surface_df = surface_df.copy()

        K = surface_df["strike"].to_numpy(dtype=float)
        T = surface_df["T"].to_numpy(dtype=float)

        # Remove invalid observations
        valid = (
            np.isfinite(K)
            & np.isfinite(T)
            & (K > 0)
            & (T > 0)
            & np.isfinite(surface_df["total_variance"])
            & (surface_df["total_variance"] > 0)
        )

        surface_df = surface_df.loc[valid].copy()

        K = surface_df["strike"].to_numpy(dtype=float)
        T = surface_df["T"].to_numpy(dtype=float)

        # Log-moneyness
        k = np.log(K / weighted_spot)

        # Avoid zero std
        k_std = k.std()
        T_std = T.std()

        if k_std == 0 or T_std == 0:
            raise ValueError("Not enough strike/maturity variation to build surface.")

        k_mean = k.mean()
        T_mean = T.mean()

        k_scaled = (k - k_mean) / k_std
        T_scaled = (T - T_mean) / T_std

        query_points = np.column_stack([k_scaled, T_scaled])

        # Interpolate log(total variance)
        log_variance = np.log(surface_df["total_variance"].to_numpy(dtype=float))

        f_variance = RBFInterpolator(
            query_points,
            log_variance,
            kernel="thin_plate_spline",
            smoothing=0.001
        )

        params = {
            "k_mean": k_mean,
            "k_std": k_std,
            "T_mean": T_mean,
            "T_std": T_std,
        }

        return f_variance, params


    # --------------------------------------------------------
    # Get IV from variance surface
    # --------------------------------------------------------

    def get_iv_from_surface(self, weighted_spot, f_variance, params, required_strike, T):

        k_mean = params["k_mean"]
        k_std = params["k_std"]
        T_mean = params["T_mean"]
        T_std = params["T_std"]

        if T <= 0:
            return np.nan

        if k_std == 0 or T_std == 0:
            return np.nan

        k = np.log(required_strike / weighted_spot)

        k_scaled = (k - k_mean) / k_std
        T_scaled = (T - T_mean) / T_std

        query_points = np.array([[k_scaled, T_scaled]])

        log_variance = f_variance(query_points)[0]
        total_variance = np.exp(log_variance)

        iv = np.sqrt(total_variance / T)

        return float(iv)


    # --------------------------------------------------------
    # Probability BTC touches above strike
    # --------------------------------------------------------

    def prob_touch_above(self, spot, required_strike, iv, T, r=0, q=0):
    
        # P(max St​ >= L), non symmetrical due to drift
        mu = r - q - 0.5 * (iv ** 2)

        d1 = (mu * T - np.log(required_strike / spot)) / (iv * np.sqrt(T))
        d2 = (-mu * T - np.log(required_strike / spot)) / (iv * np.sqrt(T))

        p_touch_above = norm.cdf(d1) + (required_strike / spot) ** ((2 * mu) / (iv ** 2)) * norm.cdf(d2)

        return p_touch_above


    def prob_touch_below(self, spot, required_strike, iv, T, r=0, q=0):

        #coco prob exit P(min St​ ≤ L)
        mu = r - q - 0.5 * (iv ** 2)

        d1 = (np.log(required_strike / spot) - mu * T) / (iv * np.sqrt(T))
        d2 = (np.log(required_strike / spot) + mu * T) / (iv * np.sqrt(T))

        p_touch_below = norm.cdf(d1) + (required_strike / spot) ** ((2 * mu) / (iv ** 2)) * norm.cdf(d2)

        return p_touch_below


# ============================================================
# OPTIONS DATA PREPARATION
# ============================================================

def prepare_options_data(options_df, timestamp_col="timestamp", expiry_col="expiry_dt", strike_col="strike", iv_col="iv", spot_col="spot"):
    """
    Convert raw options data into the fields required
    by the variance surface.

    Assumes IV is decimal:
        0.50 = 50%

    """

    df = options_df.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], utc=True)
    df[expiry_col] = pd.to_datetime(df[expiry_col], utc=True)
    df[strike_col] = pd.to_numeric(df[strike_col], errors="coerce")
    df[iv_col] = pd.to_numeric(df[iv_col], errors="coerce")
    df[spot_col] = pd.to_numeric(df[spot_col], errors="coerce")

    # Time to expiry in years
    df["T"] = ((df[expiry_col] - df[timestamp_col]).dt.total_seconds() / (365.25 * 24 * 3600))

    # Total variance
    df["total_variance"] = (df[iv_col] ** 2 * df["T"])

    # Standardize column names
    df = df.rename(columns={
        timestamp_col: "timestamp",
        expiry_col: "expiry_dt",
        strike_col: "strike",
        spot_col: "spot",
    })

    df = df[
        [
            "timestamp",
            "expiry_dt",
            "strike",
            "spot",
            "iv",
            "T",
            "total_variance",
        ]
    ]

    # Remove invalid observations
    df = df[
        np.isfinite(df["strike"])
        & np.isfinite(df["spot"])
        & np.isfinite(df["iv"])
        & np.isfinite(df["T"])
        & (df["strike"] > 0)
        & (df["spot"] > 0)
        & (df["iv"] > 0)
        & (df["T"] > 0)
    ]

    return df


# ============================================================
# GET OPTIONS SURFACE FOR A PARTICULAR PM TIMESTAMP
# ============================================================
def get_surface_at_timestamp(
    options_df,
    pm_timestamp,
    lookback_minutes=60,
):
    pm_timestamp = pd.Timestamp(pm_timestamp)

    start = pm_timestamp - pd.Timedelta(
        minutes=lookback_minutes
    )

    available = options_df[
        (options_df["timestamp"] >= start) &
        (options_df["timestamp"] <= pm_timestamp)
    ].copy()

    if available.empty:
        return None

    # Latest observation for each strike
    surface = (
        available
        .sort_values("timestamp")
        .groupby("strike", as_index=False)
        .tail(1)
        .copy()
    )

    # Age of each option observation
    surface["age_minutes"] = (
        pm_timestamp - surface["timestamp"]
    ).dt.total_seconds() / 60

    return surface

# ============================================================
# BUILD MODEL PROBABILITY FOR ONE PM OBSERVATION
# ============================================================

def get_target_iv(
    model,
    expiry_surface,
    spot,
    target_strike,
    T
    ):
        if expiry_surface is None or expiry_surface.empty:
            return np.nan, "no_surface"

        # ------------------------------------------------
        # 1. Direct observation exists
        # ------------------------------------------------

        exact = expiry_surface[
            np.isclose(
                expiry_surface["strike"],
                target_strike,
                rtol=0,
                atol=1.0,
            )
        ]

        if not exact.empty:

            obs = (
                exact
                .sort_values("timestamp")
                .iloc[-1]
            )

            iv = float(obs["iv"])

            if np.isfinite(iv) and 0 < iv < 5:
                return iv, "observed"

            return np.nan, "invalid_observed_iv"

        # ------------------------------------------------
        # 2. Need interpolation
        # ------------------------------------------------

        strikes = np.sort(
            expiry_surface["strike"]
            .dropna()
            .unique()
        )

        if len(strikes) < 3:
            return np.nan, "not_enough_strikes"

        # Do NOT extrapolate
        if (
            target_strike < strikes.min()
            or target_strike > strikes.max()
        ):
            return np.nan, "outside_range"

        # ------------------------------------------------
        # 3. Interpolate
        # ------------------------------------------------

        try:

            f_variance, params = (
                model.build_variance_surface(
                    weighted_spot=spot,
                    surface_df=expiry_surface,
                )
            )

            iv = model.get_iv_from_surface(
                weighted_spot=spot,
                f_variance=f_variance,
                params=params,
                required_strike=target_strike,
                T=T,
            )

        except Exception:
            return np.nan, "interpolation_failed"

        if not np.isfinite(iv):
            return np.nan, "invalid_interpolated_iv"

        if iv <= 0 or iv > 5:
            return np.nan, "implausible_interpolated_iv"

        return float(iv), "interpolated"


def calculate_contract_model_probability(model,
        contract,
        options_df,
        lookback_minutes=60,
        risk_free_rate=0.0,
        dividend_yield=0.0,
    ):
    """
    Calculate model probability for ONE synthetic BTC
    barrier-touch contract.

    Contract columns required:

        timestamp
        spot
        direction
        barrier
        expiry

    Returns a dictionary containing:

        model_prob
        iv_target
        T
        spot
        barrier
        direction
    """

    timestamp = pd.Timestamp(
        contract["timestamp"]
    )

    spot = float(contract["spot"])

    barrier = float(contract["barrier"])

    direction = contract["direction"]

    expiry = pd.Timestamp(
        contract["expiry"]
    )

    # --------------------------------------------------------
    # Time to synthetic contract expiry
    # --------------------------------------------------------

    T = (
        expiry - timestamp
    ).total_seconds() / (
        365.25 * 24 * 3600
    )

    if T <= 0:
        return None


    # --------------------------------------------------------
    # Get options surface available at that timestamp
    # --------------------------------------------------------

    surface = get_surface_at_timestamp(
        options_df=options_df,
        pm_timestamp=timestamp,
        lookback_minutes=lookback_minutes,
    )

    if surface is None or surface.empty:
        return None

    # --------------------------------------------------------
    # Make sure barrier is on the correct side
    # --------------------------------------------------------

    if direction == "up" and barrier <= spot:
        return None

    if direction == "down" and barrier >= spot:
        return None


    # --------------------------------------------------------
    # Build variance surface
    # --------------------------------------------------------
    try:
        iv, iv_source = get_target_iv(model=model, expiry_surface=surface, spot=spot, target_strike=barrier, T=T)
    except:
        return None

    if not np.isfinite(iv):
        return None

    # --------------------------------------------------------
    # Calculate touch probability
    # --------------------------------------------------------

    if direction == "up":

        model_prob = model.prob_touch_above(
            spot=spot,
            required_strike=barrier,
            iv=iv,
            T=T,
            r=risk_free_rate,
            q=dividend_yield,
        )

    elif direction == "down":

        model_prob = model.prob_touch_below(
            spot=spot,
            required_strike=barrier,
            iv=iv,
            T=T,
            r=risk_free_rate,
            q=dividend_yield,
        )

    else:
        raise ValueError(
            f"Unknown direction: {direction}"
        )

    if not np.isfinite(model_prob):
        return None

    return {
        "model_prob": model_prob,
        "iv_target": iv,
        "iv_source": iv_source,
        "T": T,
        "spot": spot,
        "barrier": barrier,
        "direction": direction,
        "timestamp": timestamp,
        "expiry": expiry,
        "options_timestamp": surface["timestamp"].max(),
    }


model = OptionSurface()

model_results = []

for i, row in eth_realized_df.iterrows():

    if i % 1000 == 0:
        print(i, "/", len(eth_realized_df))

    result = calculate_contract_model_probability(
        model=model,
        contract=row,
        options_df=eth_options,
        lookback_minutes=60,
    )

    if result is None:

        model_results.append({
            "model_prob": np.nan,
            "iv_target": np.nan,
            "iv_source": None,
            "T": np.nan,
            "options_timestamp": pd.NaT,
        })

    else:

        model_results.append({
            "model_prob": result["model_prob"],
            "iv_target": result["iv_target"],
            "iv_source": result["iv_source"],
            "T": result["T"],
            "options_timestamp": result["options_timestamp"],
        })

model_results = pd.DataFrame(
    model_results,
    index=eth_realized_df.index
)

eth_result = pd.concat(
    [
        eth_realized_df,
        model_results
    ],
    axis=1
)

eth_result

0 / 191614
1000 / 191614
2000 / 191614
3000 / 191614
4000 / 191614
5000 / 191614
6000 / 191614
7000 / 191614
8000 / 191614
9000 / 191614
10000 / 191614
11000 / 191614
12000 / 191614
13000 / 191614
14000 / 191614
15000 / 191614
16000 / 191614
17000 / 191614
18000 / 191614
19000 / 191614
20000 / 191614
21000 / 191614
22000 / 191614
23000 / 191614
24000 / 191614
25000 / 191614
26000 / 191614
27000 / 191614
28000 / 191614
29000 / 191614
30000 / 191614
31000 / 191614
32000 / 191614
33000 / 191614
34000 / 191614
35000 / 191614
36000 / 191614
37000 / 191614
38000 / 191614
39000 / 191614
40000 / 191614
41000 / 191614
42000 / 191614
43000 / 191614
44000 / 191614
45000 / 191614
46000 / 191614
47000 / 191614
48000 / 191614
49000 / 191614
50000 / 191614
51000 / 191614
52000 / 191614
53000 / 191614
54000 / 191614
55000 / 191614
56000 / 191614
57000 / 191614
58000 / 191614
59000 / 191614
60000 / 191614
61000 / 191614
62000 / 191614
63000 / 191614
64000 / 191614
65000 / 191614
66000 / 191614
67000 / 

,timestamp,spot,direction,barrier,horizon,expiry,realized_touch,model_prob,iv_target,iv_source,T,options_timestamp
0,2026-01-11 12:00:00+00:00,3107.69,down,1000.0,7d,2026-01-18 12:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT
1,2026-01-11 12:00:00+00:00,3107.69,down,1200.0,7d,2026-01-18 12:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT
2,2026-01-11 12:00:00+00:00,3107.69,down,1400.0,7d,2026-01-18 12:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT
3,2026-01-11 12:00:00+00:00,3107.69,down,1600.0,7d,2026-01-18 12:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT
4,2026-01-11 12:00:00+00:00,3107.69,down,1800.0,7d,2026-01-18 12:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...
191609,2026-07-24 23:00:00+00:00,1860.82,up,3200.0,7d,2026-07-31 23:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT
191610,2026-07-24 23:00:00+00:00,1860.82,up,3400.0,7d,2026-07-31 23:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT
191611,2026-07-24 23:00:00+00:00,1860.82,up,3600.0,7d,2026-07-31 23:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT
191612,2026-07-24 23:00:00+00:00,1860.82,up,3800.0,7d,2026-07-31 23:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT


In [ ]:
model = OptionSurface()

model_results = []

for i, row in btc_realized_df.iterrows():

    if i % 1000 == 0:
        print(i, "/", len(btc_realized_df))

    result = calculate_contract_model_probability(
        model=model,
        contract=row,
        options_df=btc_options,
        lookback_minutes=60,
    )

    if result is None:

        model_results.append({
            "model_prob": np.nan,
            "iv_target": np.nan,
            "iv_source": None,
            "T": np.nan,
            "options_timestamp": pd.NaT,
        })

    else:

        model_results.append({
            "model_prob": result["model_prob"],
            "iv_target": result["iv_target"],
            "iv_source": result["iv_source"],
            "T": result["T"],
            "options_timestamp": result["options_timestamp"],
        })

model_results = pd.DataFrame(
    model_results,
    index=btc_realized_df.index
)

btc_result = pd.concat(
    [
        btc_realized_df,
        model_results
    ],
    axis=1
)

btc_result

0 / 232973
1000 / 232973
2000 / 232973
3000 / 232973
4000 / 232973
5000 / 232973
6000 / 232973
7000 / 232973
8000 / 232973
9000 / 232973
10000 / 232973
11000 / 232973
12000 / 232973
13000 / 232973
14000 / 232973
15000 / 232973
16000 / 232973
17000 / 232973
18000 / 232973
19000 / 232973
20000 / 232973
21000 / 232973
22000 / 232973
23000 / 232973
24000 / 232973
25000 / 232973
26000 / 232973
27000 / 232973
28000 / 232973
29000 / 232973
30000 / 232973
31000 / 232973
32000 / 232973
33000 / 232973
34000 / 232973
35000 / 232973
36000 / 232973
37000 / 232973
38000 / 232973
39000 / 232973
40000 / 232973
41000 / 232973
42000 / 232973
43000 / 232973
44000 / 232973
45000 / 232973
46000 / 232973
47000 / 232973
48000 / 232973
49000 / 232973
50000 / 232973
51000 / 232973
52000 / 232973
53000 / 232973
54000 / 232973
55000 / 232973
56000 / 232973
57000 / 232973
58000 / 232973
59000 / 232973
60000 / 232973
61000 / 232973
62000 / 232973
63000 / 232973
64000 / 232973
65000 / 232973
66000 / 232973
67000 / 

,timestamp,spot,direction,barrier,horizon,expiry,realized_touch,model_prob,iv_target,iv_source,T,options_timestamp
0,2026-01-13 02:00:00+00:00,91207.89,down,50000.0,7d,2026-01-20 02:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT
1,2026-01-13 02:00:00+00:00,91207.89,down,60000.0,7d,2026-01-20 02:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT
2,2026-01-13 02:00:00+00:00,91207.89,down,70000.0,7d,2026-01-20 02:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT
3,2026-01-13 02:00:00+00:00,91207.89,down,80000.0,7d,2026-01-20 02:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT
4,2026-01-13 02:00:00+00:00,91207.89,down,90000.0,7d,2026-01-20 02:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...
232968,2026-08-21 00:00:00+00:00,73423.20,up,160000.0,7d,2026-08-28 00:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT
232969,2026-08-21 00:00:00+00:00,73423.20,up,170000.0,7d,2026-08-28 00:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT
232970,2026-08-21 00:00:00+00:00,73423.20,up,180000.0,7d,2026-08-28 00:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT
232971,2026-08-21 00:00:00+00:00,73423.20,up,190000.0,7d,2026-08-28 00:00:00+00:00,0,NaN,NaN,NaN,NaN,NaT


In [ ]:
btc_result = pd.read_parquet("btc_result.parquet")
eth_result = pd.read_parquet("eth_result.parquet")

In [96]:
btc_result1 = btc_result[btc_result["T"] > 0].copy()

btc_result1["prob_bucket"] = pd.cut(
    btc_result1["model_prob"],
    bins=np.arange(0, 1.01, 0.10),
    include_lowest=True
)

btc_calibration = (
    btc_result1
    .dropna(subset=["model_prob", "realized_touch"])
    .groupby("prob_bucket", observed=True)
    .agg(
        n=("realized_touch", "size"),
        predicted_prob=("model_prob", "mean"),
        actual_touch_rate=("realized_touch", "mean"),
    )
    .reset_index()
)

btc_calibration["calibration_error"] = (
    btc_calibration["actual_touch_rate"]
    - btc_calibration["predicted_prob"]
)

btc_calibration

,prob_bucket,n,predicted_prob,actual_touch_rate,calibration_error
0,"(-0.001, 0.1]",13893,0.016247,0.037285,0.021038
1,"(0.1, 0.2]",2020,0.145802,0.071287,-0.074515
2,"(0.2, 0.3]",1568,0.249666,0.230867,-0.018798
3,"(0.3, 0.4]",1327,0.346938,0.318011,-0.028928
4,"(0.4, 0.5]",1053,0.448275,0.400760,-0.047515
5,"(0.5, 0.6]",890,0.547940,0.475281,-0.072659
6,"(0.6, 0.7]",840,0.648992,0.632143,-0.016849
7,"(0.7, 0.8]",690,0.748281,0.676812,-0.071469
8,"(0.8, 0.9]",683,0.848801,0.777452,-0.071348
9,"(0.9, 1.0]",830,0.951808,0.921687,-0.030122


In [88]:
eth_result1 = eth_result[eth_result["T"] > 0].copy()

eth_result1["prob_bucket"] = pd.cut(
    eth_result1["model_prob"],
    bins=np.arange(0, 1.01, 0.10),
    include_lowest=True
)

eth_calibration = (
    eth_result1
    .dropna(subset=["model_prob", "realized_touch"])
    .groupby("prob_bucket", observed=True)
    .agg(
        n=("realized_touch", "size"),
        predicted_prob=("model_prob", "mean"),
        actual_touch_rate=("realized_touch", "mean"),
    )
    .reset_index()
)

eth_calibration["calibration_error"] = (
    eth_calibration["actual_touch_rate"]
    - eth_calibration["predicted_prob"]
)

eth_calibration

,prob_bucket,n,predicted_prob,actual_touch_rate,calibration_error
0,"(-0.001, 0.1]",13351,0.014726,0.015654,0.000929
1,"(0.1, 0.2]",2030,0.148177,0.103941,-0.044236
2,"(0.2, 0.3]",1065,0.245817,0.153991,-0.091826
3,"(0.3, 0.4]",899,0.347345,0.290323,-0.057022
4,"(0.4, 0.5]",839,0.450329,0.456496,0.006166
5,"(0.5, 0.6]",665,0.546159,0.463158,-0.083001
6,"(0.6, 0.7]",664,0.650631,0.546687,-0.103944
7,"(0.7, 0.8]",553,0.750034,0.670886,-0.079148
8,"(0.8, 0.9]",680,0.849264,0.779412,-0.069852
9,"(0.9, 1.0]",672,0.949032,0.924107,-0.024925
